# 02 · Data preparation

next-sequence trains on **DataLoaders that you build**. This notebook shows how the data must look
and the order of the steps:

```
raw events → 1. one sample per sequence → 2. split (by time or user)
           → 3. fit vocabulary + scalers on TRAIN → 4. transform every split
           → 5. one collate function per feature → 6. CustomCollate → DataLoader
```

## The example data

Every cookbook uses the same small synthetic shop: each row is one purchase with an `item`,
a `price` and a `weekday`. Shoppers buy the five items in a fixed cycle
(`apple → bread → cheese → dates → eggs → apple ...`), starting at a random item, so the next
item is learnable and we can check the model gets it right.

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

events = make_events()
print(events.shape)
events.head(8)

(2676, 5)


,user_id,t,item,price,weekday
0,0,0,dates,102.41,0
1,0,1,eggs,57.11,4
2,0,2,apple,19.94,6
3,0,3,bread,47.97,3
4,0,4,cheese,174.89,1
5,0,5,dates,95.09,2
6,0,6,eggs,55.68,0
7,0,7,apple,20.47,2


## 1. One sample per sequence

Each sample is a dict. Every sequence feature is a **list ordered by time** (oldest first), and all
lists in a sample have the same length. Extra keys such as `user_id` are fine; only keys named by your
`Feature`s are used.

In [2]:
users = (events.sort_values(["user_id", "t"])
               .groupby("user_id").agg(list).reset_index()
               .drop(columns="t").to_dict(orient="records"))
print(len(users), "sequences")
{k: (v[:5] + ["..."] if isinstance(v, list) else v) for k, v in users[0].items()}   # first 5 steps

300 sequences


{'user_id': 0,
 'item': ['dates', 'eggs', 'apple', 'bread', 'cheese', '...'],
 'price': [102.41, 57.11, 19.94, 47.97, 174.89, '...'],
 'weekday': [0, 4, 6, 3, 1, '...']}

## 2. Split **before** fitting anything

Split by **time** or by **user**, never by row. One history produces many near-identical prefixes
(`[a]→b`, `[a,b]→c`, ...); if they end up in both train and validation, the validation score is
much better than reality.

**Leave-last-out** (by time) is three slices per user. The last element of each list is the target:

In [3]:
def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]   # predicts the 3rd-to-last event
val_raw = [cut(u, -1) for u in users]     # predicts the 2nd-to-last event
test_raw = [cut(u, None) for u in users]  # predicts the last event

u = users[0]["item"]
pd.DataFrame({"split": ["train", "val", "test"],
              "input": [u[:-3], u[:-2], u[:-1]],
              "target": [u[-3], u[-2], u[-1]]})

,split,input,target
0,train,"[dates, eggs, apple, bread, cheese, dates, egg...",cheese
1,val,"[dates, eggs, apple, bread, cheese, dates, egg...",dates
2,test,"[dates, eggs, apple, bread, cheese, dates, egg...",eggs


## 3. Fit the vocabulary and scalers on the training split only

`create_bow` gives deterministic ids: `PAD=0`, `UNK=1`, then the tokens sorted.

In [4]:
import math
from next_sequence import create_bow

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5
print("vocabulary:", bow)
print(f"log1p(price): mean={price_mean:.3f} std={price_std:.3f}")

vocabulary: {'PAD': 0, 'UNK': 1, 'apple': 2, 'bread': 3, 'cheese': 4, 'dates': 5, 'eggs': 6}
log1p(price): mean=4.154 std=0.725


## 4. Transform every split with the same preprocessing

- Unseen tokens map to `UNK`.
- **Id 0 is padding**: shift categorical ids that start at 0 (weekday `0..6` → `1..7`).
- **Scale numeric values.** Raw prices would give an MSE in the thousands that drowns the other tasks.

In [5]:
def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

raw_prices = pd.Series([p for u in train_raw for p in u["price"]])
scaled = pd.Series([p for u in train for p in u["price"]])
pd.DataFrame({"raw price": raw_prices.describe(), "scaled price": scaled.describe()}).round(2)

,raw price,scaled price
count,2076.00,2076.00
mean,80.58,-0.00
std,56.20,1.00
min,18.00,-1.67
25%,42.81,-0.52
50%,60.04,-0.06
75%,99.67,0.63
max,197.90,1.57


In [6]:
# an item that was never seen in training becomes UNK
transform([{"item": ["apple", "durian"], "price": [20.0, 99.0], "weekday": [0, 6]}])

[{'item': [2, 1],
  'price': [-1.5295354151404883, 0.6219108871529243],
  'weekday': [1, 7]}]

## 5. One collate function per feature

`CustomCollate` calls `funcs[name](values, inference)` with that feature's values for the whole batch.
`make_collate` builds such a function, with both branches from the same code:

- training (`inference=False`): inputs are every step but the last, the target is the last step
- inference (`inference=True`): inputs are the full history, no target

In [7]:
from next_sequence import check_collate, make_collate

funcs = {"item": make_collate("long"), "price": make_collate("float"), "weekday": make_collate("long")}
print(funcs["item"]([[2, 3, 4], [5, 6]]))
check_collate(funcs["item"], [u["item"] for u in train[:3]])

([tensor([2, 3]), tensor([5])], [4, 6])


{'input_dtype': torch.int64, 'targets': [4, 6, 3]}

> **Other targets** (the change since the last step, buckets, ratios, your own rule):
> see **03_custom_collate_functions**.

## 6. `CustomCollate`: from samples to a padded batch

`CustomCollate.from_config(config, ...)` copies `max_len` and the input order from the model
configuration, so we create the configuration first.

In [8]:
from next_sequence import CustomCollate, Feature, NextSequenceConfig, TaskType

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)
collate = CustomCollate.from_config(config, feature_mask="item", funcs=funcs)

print("first sample, oldest first:", train[0]["item"])
batch = collate(train[:3])
print("lengths:", batch["lengths"].tolist())
print("X['item'] (newest step first):\n", batch["X"]["item"])
print("X['price'] shape:", tuple(batch["X"]["price"].shape), "(numeric features get a trailing 1)")
print("mask (True = padding):\n", batch["mask"])
print("Y:", {k: [round(v, 3) for v in t.tolist()] for k, t in batch["Y"].items()})

first sample, oldest first: [5, 6, 2, 3, 4, 5, 6, 2, 3, 4]
lengths: [9, 9, 4]
X['item'] (newest step first):
 tensor([[3, 2, 6, 5, 4, 3, 2, 6, 5],
        [5, 4, 3, 2, 6, 5, 4, 3, 2],
        [2, 6, 5, 4, 0, 0, 0, 0, 0]])
X['price'] shape: (3, 9, 1) (numeric features get a trailing 1)
mask (True = padding):
 tensor([[False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False],
        [False, False, False, False,  True,  True,  True,  True,  True]])
Y: {'item': [4, 6, 3], 'price': [1.566, -0.087, -0.369], 'weekday': [7, 7, 4]}


- `feature_mask` names the feature whose lengths define the padding mask.
- **Inputs come newest step first** (`recent_first=True` in the configuration): the latest event
  always sits right next to the model's summary token, which makes "what happened last" easy to
  learn. You still store your lists oldest first; the collate reverses them. Padding stays on the right.

### Inference mode: the full history, no targets

In [9]:
inference_collate = CustomCollate.from_config(config, "item", funcs, inference=True)
b = inference_collate(test[:3])
print("lengths:", b["lengths"].tolist(), "| Y:", b["Y"])

lengths: [12, 12, 7] | Y: None


### `max_len` keeps the most recent steps

Histories longer than `max_len` are cut from the left, so the newest events are kept.

In [10]:
short_config = NextSequenceConfig(features=features, max_len=4, emb_dim=32, attn_heads=2)
short = CustomCollate.from_config(short_config, "item", funcs, inference=True)
print("full history (oldest first):", test[0]["item"])
print("kept, newest first         :", short(test[:1])["X"]["item"][0].tolist())

full history (oldest first): [5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6]
kept, newest first         : [6, 5, 4, 3]


### Mistakes in the data are reported clearly

In [11]:
broken = [{"item": [2, 3, 4], "price": [0.1, 0.2], "weekday": [1, 2, 3]}]   # price is one step short
try:
    collate(broken)
except ValueError as e:
    print("ValueError:", e)

ValueError: sample 0: 'price' has 1 input steps but 'item' has 2; all features of a sample must have the same length


## 7. DataLoaders

In [12]:
from torch.utils.data import DataLoader
from next_sequence import CoreDataset

train_loader = DataLoader(CoreDataset(train), batch_size=32, shuffle=True, collate_fn=collate)  # shuffle train only
val_loader = DataLoader(CoreDataset(val), batch_size=32, collate_fn=collate)
test_loader = DataLoader(CoreDataset(test), batch_size=32, collate_fn=collate)
print(len(train_loader), "training batches")

10 training batches


## Checklist

- [ ] Split by time or user **before** building the vocabulary and scalers
- [ ] Unseen tokens map to `UNK`; categorical ids start at 1 (0 is PAD); class targets start at 0
- [ ] Numeric inputs and regression targets are scaled
- [ ] Collate functions come from `make_collate` (check custom ones with `check_collate`)
- [ ] Build collates with `CustomCollate.from_config(config, ...)`
- [ ] Only the training loader shuffles